In [ ]:
def post_process_results():

    '''This script post processes the results of an optimization run to extract parameters of interest'''

    # Define global variables
    
    global x_b_res, indices_res, x_b, residential_retrofit_1, residential_retrofit_2
    global nonres_retrofit_1, nonres_retrofit_2, total_spending, sheltered_homes,  investment_total, res_shelt_perc
    global res_1_perc, res_2_perc, nonres_1_perc, nonres_2_perc, perc_retro_res, savings, total_cost, BCR
    global bld_cost, dislocation_cost, debris_cost_bld, casualty_cost, perc_retro
    global bld_cost_0, dislocation_cost_0, debris_cost_bld_0, casualty_cost_0,  bld_cost_0

    # Get building counts
    x_b = ampl.get_variable('x_b').to_pandas().unstack().values.reshape(x_b_0.shape)

    # deal with rounding issue
    x_b[x_b<0.1] = 0
    
    # ====================== Residential =======================
    
    x_b_res = x_b[:,np.where(indices_noret['Class']=='Res')][:,0,:] # residential building count matrix
    indices_res = indices_noret[indices_noret['Class']=='Res'].reset_index() # indices of residential locations
    
    # Update neighborhoods with residential building retrofit, shelter, or both
    
    neighborhoods['retrofit'] = x_b_res[:,:,[1,2]].sum(axis=1).sum(axis=1)/x_b_res.sum(axis=1).sum(axis=1)*100
    neighborhoods['shelter'] = x_b_res[:,:,3].sum(axis=1)/x_b_res.sum(axis=1).sum(axis=1)*100
    neighborhoods['both'] = x_b_res[:,:,4:].sum(axis=1).sum(axis=1)/x_b_res.sum(axis=1).sum(axis=1)*100
        
    # Get percentages of different retrofits (percent sheltered, upgrade 1, upgrade 2, shelter + upgrade 1/2, none
    
    perc_retro_res = np.empty(6) # percent of homes in a given retrofit state 0 to 5
    for i in np.arange(perc_retro_res.shape[0]):
        perc_retro_res[i] = x_b_res[:,:,i].sum()/x_b_res.sum(axis=1).sum()

    perc_retro = np.empty(6) # percent of homes in a given retrofit state 0 to 5
    for i in np.arange(perc_retro.shape[0]):
        perc_retro[i] = x_b[:,:,i].sum()/x_b.sum(axis=1).sum()

    
    # Any upgrade 1 or 2 (with or without shelter) or any shelter
    
    residential_retrofit_1 = x_b_res[:,:,[1,4]].sum()
    residential_retrofit_2 = x_b_res[:,:,[2,5]].sum()
    
    sheltered_homes = x_b_res[:,:,3:].sum()
    
    res_shelt_perc = sheltered_homes/x_b_res.sum()*100
    
    res_1_perc = residential_retrofit_1/x_b_res.sum()*100
    res_2_perc = residential_retrofit_2/x_b_res.sum()*100
    
    # =========================== Any building ===========
    
    # Update neighborhood with percent of any retrofit
    
    neighborhoods['retro_any'] = x_b[:,:,1:].sum(axis=1).sum(axis=1)/x_b.sum(axis=1).sum(axis=1)*100
    
    # Any upgrade 1 or 2 for a nonresidential
    
    nonres_retrofit_1 =  x_b[:,np.where(indices_noret['Class']!='Res')[0].tolist(),:][:,:,[1,4]].sum()
    nonres_retrofit_2 =  x_b[:,np.where(indices_noret['Class']!='Res')[0].tolist(),:][:,:,[2,5]].sum()
    
    nonres_1_perc = nonres_retrofit_1/x_b[:,np.where(indices_noret['Class']!='Res')[0].tolist(),:].sum()*100
    nonres_2_perc = nonres_retrofit_2/x_b[:,np.where(indices_noret['Class']!='Res')[0].tolist(),:].sum()*100
    
    # ==================== Costs, savings, and budgets=========================================
    
    total_spending = ampl.get_variable('c_ret').get_values().to_pandas().unstack().values[0]/10**6 
    
    # ======= Social/demographic
    neighborhoods['%Minority'] = neighborhoods['PercAA'] + neighborhoods['PercNA'] + neighborhoods['PercHisp']
        
    # Get costs per neighborhood for plotting (original and optimal)
    
    bld_cost = f_pw*ampl.get_data('{n in N} sum{a in A, k in K, r in R} x_b[n,a,r]*c_b_rep[a,k]*p_DS[n,a,r,k]*f_footprint[a]').to_pandas().values[:,0]
    bld_cost_0 = f_pw*ampl.get_data('{n in N} sum{a in A, k in K, r in R} x_b_0[n,a,r]*c_b_rep[a,k]*p_DS[n,a,r,k]*f_footprint[a]').to_pandas().values[:,0]
    
    dislocation_cost = f_pw*ampl.get_data('{n in N} sum{a in A,r in R} x_b[n,a,r]*c_d[n,a,r]*f_footprint[a]').to_pandas().values[:,0]
    dislocation_cost_0 = f_pw*ampl.get_data('{n in N} sum{a in A,r in R} x_b_0[n,a,r]*c_d[n,a,r]*f_footprint[a]').to_pandas().values[:,0]
    
    debris_cost_bld = f_pw*ampl.get_data('{n in N} sum{a in A, r in R} x_b[n,a,r]*V_db[n,a,r]*C_dr*f_footprint[a]').to_pandas().values[:,0]
    debris_cost_bld_0 = f_pw*ampl.get_data('{n in N} sum{a in A, r in R} x_b_0[n,a,r]*V_db[n,a,r]*C_dr*f_footprint[a]').to_pandas().values[:,0]
    
    casualty_cost = f_pw*ampl.get_data('{n in N} sum{ m in M, a in A, k in K, r in R} c_cas[m]*p_i[k,m]*p_DS_for_casualties[n,a,r,k]*H[n,a]*x_b[n,a,r]*f_footprint[a]').to_pandas().values[:,0]
    casualty_cost_0 = f_pw*ampl.get_data('{n in N} sum{ m in M, a in A, k in K, r in R} c_cas[m]*p_i[k,m]*p_DS_for_casualties[n,a,r,k]*H[n,a]*x_b_0[n,a,r]*f_footprint[a]').to_pandas().values[:,0]
            
    total_cost = bld_cost + casualty_cost + dislocation_cost + debris_cost_bld 
    
    # get savings, CBR, investment
    
    investment_total = ampl.get_data('{n in N} sum{ a in A, r in R, r_prime in R:(a,r,r_prime) in Q} change_matrix[n,a,r,r_prime]*c_b_ret[a,r,r_prime]').to_pandas().values[:,0]

    # savings
    savings = (bld_cost_0-bld_cost)+(dislocation_cost_0-dislocation_cost)+(debris_cost_bld_0-debris_cost_bld)+(casualty_cost_0-casualty_cost)

    # benefit-to-cost ratio
    BCR =  savings/investment_total
    BCR[investment_total==0]=0
    
    neighborhoods['BCR'] = np.array(BCR)

    neighborhoods[['BCR','retro_any','retrofit','shelter','both']] = neighborhoods[['BCR','retro_any','retrofit','shelter','both']].fillna(0)